# L00 · 环境自检

**预计时长**：20 min
**难度**：⭐
**前置**：无（第一节课）

## 本节目标

1. 确认 Python 环境、必需库版本符合要求
2. 确认 akshare 能拉到真实 A 股数据
3. 确认 Matplotlib 中文显示正常（不会出方块字）
4. 把比亚迪数据缓存到本地 parquet，后续课程秒级读取

## 学习方法提示

- 每节 notebook 按 7 段结构组织：**元信息 → 概念 → 数据 → 演示 → 小练 → 习题 → Jupyter tip**
- 代码格**逐个运行**（Shift+Enter），不要一次批量执行
- 遇到不懂的 API 立即问 Claude
- 课后练习独立完成，二次错才看答案

## 第 1 段：版本检查

需要的库与最低版本：

| 库 | 用途 | 最低版本 |
|----|------|---------|
| python | 运行时 | 3.9 |
| pandas | 表格处理 | 2.0 |
| numpy | 数值计算 | 1.24 |
| matplotlib | 绘图 | 3.7 |
| akshare | A 股数据源 | 1.12 |
| jupyterlab | notebook 环境 | 4.0 |
| mplfinance | 专业 K 线图（L02 用） | 0.12 |
| seaborn | 统计图（L06 用） | 0.13 |
| pyarrow | parquet 读写 | 14.0 |

In [ ]:
import sys
import importlib

REQUIRED = {
    "python": (3, 9),
    "pandas": (2, 0),
    "numpy": (1, 24),
    "matplotlib": (3, 7),
    "akshare": (1, 12),
    "jupyterlab": (4, 0),
    "mplfinance": (0, 12),
    "seaborn": (0, 13),
    "pyarrow": (14, 0),
}

def version_tuple(name: str) -> tuple[int, int]:
    if name == "python":
        return sys.version_info[:2]
    mod = importlib.import_module(name)
    v = getattr(mod, "__version__", "0.0")
    parts = v.split(".")[:2]
    try:
        return tuple(int(p) for p in parts)
    except ValueError:
        return (0, 0)

results = []
for name, min_v in REQUIRED.items():
    try:
        actual = version_tuple(name)
        ok = actual >= min_v
        results.append((name, ".".join(map(str, actual)), ".".join(map(str, min_v)), ok))
    except Exception as e:
        results.append((name, "MISSING", ".".join(map(str, min_v)), False))

print(f"{'库':<14} {'当前':<12} {'最低':<10} {'状态'}")
print("-" * 50)
for name, cur, mn, ok in results:
    print(f"{name:<14} {cur:<12} {mn:<10} {'✅' if ok else '❌'}")

全部 ✅ 才能继续。如果有 ❌，在终端运行：

```bash
.venv/Scripts/python.exe -m pip install -r requirements.txt
```

## 第 2 段：模块导入 + 字体设置

后面所有 notebook 的第一格都长这样。`import _style` 的副作用是设中文字体。

In [ ]:
import sys
from pathlib import Path

# 自动定位 phase1_foundation 目录（无论 jupyter lab 从项目根还是从这里启动）
_cwd = Path.cwd()
_p1 = _cwd if (_cwd / '_data.py').exists() else (_cwd / 'learning' / 'phase1_foundation')
sys.path.insert(0, str(_p1))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import _style  # 副作用：设中文字体
from _data import get_stock_data

## 第 3 段：中文字体测试

如果方块字，说明系统没有 Microsoft YaHei，需要改 `_style.py` 的字体栈。

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.text(0.5, 0.7, "中文测试：股票、ETF、涨跌停板", ha='center', fontsize=20)
ax.text(0.5, 0.3, "负号测试：-123.45 +789.00", ha='center', fontsize=16)
ax.set_axis_off()
plt.show()

## 第 4 段：akshare 连通性 + 数据缓存

第一次拉取会走网络（约 3–5 秒），后续读 parquet 缓存（毫秒级）。

In [ ]:
# force_refresh=True 强制重新联网，验证 akshare 通畅
df = get_stock_data("002594", force_refresh=True)
print(f"行数: {len(df)}")
print(f"日期范围: {df['date'].min().date()} ~ {df['date'].max().date()}")
print(f"是否合成数据: {df.attrs.get('synthetic')}")
df.head()

In [ ]:
# 验证缓存文件已生成
from pathlib import Path
cache = Path("data/002594_qfq.parquet")
print(f"缓存存在: {cache.exists()}, 大小: {cache.stat().st_size // 1024} KB")

## 第 5 段：课程导览

本阶段 10 节课，每节锁定 **1 个金融概念 × 1 个数据技能**：

| # | 课题 | # | 课题 |
|---|------|---|------|
| 00 | 环境自检（本节） | 05 | 收益率 + 涨停识别 |
| 01 | 股票基础 + A 股规则 | 06 | 统计基础 + 交易成本 |
| 02 | K 线读图 + DataFrame | 07 | 技术指标入门 |
| 03 | 量价关系 + 聚合 | 08 | PE 估值 + 行业对比 |
| 04 | 数据清洗 + 复权 + 对齐 | 09 | 向量化核心习惯 |
| | | 10 | 综合项目 |

**核心目标**：L09 结束时，"向量化优先" 应该成为你的肌肉记忆；L10 结束时，你应该能独立产出一份股票分析报告。

详细大纲见 `learning/INDEX.md`。

## 第 6 段：Jupyter tip 🔧

本节用到的 Jupyter 技巧：

| 操作 | 作用 |
|------|------|
| `Shift + Enter` | 运行当前格并跳到下一格 |
| `Ctrl + Enter` | 运行当前格但不跳转 |
| `Esc` → `dd` | 删除当前格（先按 Esc 进入命令模式） |
| `Esc` → `a` / `b` | 在上方/下方插入新格 |
| `%timeit x = sum(range(1000))` | 测一行代码的执行时间（L09 会大量用到） |
| `?print` | 查看函数文档 |
| `??print` | 查看函数源码 |

**最重要的习惯**：从 L01 开始，每运行完一个 cell，看一眼输出，确认它符合你的预期。看到奇怪的数字或图，**不要跳过**，立即问。

## 收尾

如果上面 4 段全部 ✅：恭喜，你已具备学习后续 9 节课的全部环境条件。

**下一节 L01**：股票基础 + A 股规则（T+1、涨跌停板）。你会拉比亚迪/世纪华通/完美世界三只股票的数据，并在 K 线表里识别涨停日。

**课后无练习**（环境课），直接进 L01。